# A Confidence Interval With No Formula

**Topic:** Bootstrap resampling &nbsp;·&nbsp; **Difficulty:** Medium

Latency reports quote the median, not the mean, because one 40-second request
should not move the headline number. That is good statistics and awkward
engineering: the median has no tidy standard-error formula the way the mean does.

The bootstrap does not care. Resample the data with replacement a few thousand
times, take the median of each resample, and read the interval straight off the
percentiles of what comes back. Note that it will *not* come out symmetric
around the sample median, and that is correct rather than a bug.

## The data (created in the next cell)

- `latencies` `(900,)`: request latencies in milliseconds, heavily right-skewed.
- Use `np.random.default_rng(404)` for the resampling so results reproduce.

## Your task

With **NumPy only**:

1. Draw `B = 4000` bootstrap resamples of size `len(latencies)`, **with
   replacement**, and take the median of each, drawing the resample indices
   with a single `rng.integers(0, n, (B, n))` call.
2. Form the 95% percentile interval from the 2.5th and 97.5th percentiles of
   those bootstrap medians, using `np.percentile` with its default (linear)
   interpolation.
3. Set `answers` to a dict with `"sample_median"`, `"ci_low"`, `"ci_high"`,
   `"boot_se"` (the standard deviation of the bootstrap medians,
   `np.std(medians, ddof=1)`), and
   `"asymmetry"`: `(ci_high - median) - (median - ci_low)`.

## Evaluation

Compared against the reference to a relative tolerance of 1e-6. The evaluation
cell also confirms the interval brackets the sample median.

In [ ]:
import numpy as np

rng = np.random.default_rng(88)
n = 900
latencies = rng.lognormal(mean=3.2, sigma=0.75, size=n)
latencies[rng.choice(n, 20, replace=False)] *= 12.0     # a few pathological requests
print(f"n = {n}, mean = {latencies.mean():.1f} ms, median = {np.median(latencies):.1f} ms")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("sample_median", [], 24.969279604348806, -4.180806608300327, 1e-06),
    ("ci_low", [], 23.62829891913184, -3.9562754644630114, 1e-06),
    ("ci_high", [], 26.664076609652966, -4.464580054382014, 1e-06),
    ("boot_se", [], 0.7639593332053422, -0.12791583415091098, 1e-06),
    ("asymmetry", [], 0.3538163200871942, -0.0592423022443712, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "ci_low" in answers and "ci_high" in answers and "sample_median" in answers:
    _br = answers["ci_low"] <= answers["sample_median"] <= answers["ci_high"]
    print(f"  {'interval brackets median':<24} {'ok' if _br else 'wrong'}")
    _ok = _ok and _br

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Draw all `B * n` indices in one `integers` call with shape `(B, n)`, because looping 4000 times
in Python is needlessly slow. The interval is asymmetric because the sampling
distribution of the median is itself skewed when the data are. A symmetric
`estimate +- 1.96 se` interval would be wrong on the right tail.

```python
import numpy as np

rng = np.random.default_rng(404)
n = len(latencies)
B = 4000

idx = rng.integers(0, n, (B, n))
boot_medians = np.median(latencies[idx], axis=1)

sample_median = float(np.median(latencies))
ci_low = float(np.percentile(boot_medians, 2.5))
ci_high = float(np.percentile(boot_medians, 97.5))

answers = {
    "sample_median": sample_median,
    "ci_low": ci_low,
    "ci_high": ci_high,
    "boot_se": float(boot_medians.std(ddof=1)),
    "asymmetry": float((ci_high - sample_median) - (sample_median - ci_low)),
}
```